# 1. Data preparation and feature construction

This notebook follows the CS230 assignment pattern: define the task, implement each preparation stage in focused cells, check its behavior, and run the full pipeline only after reviewing the steps.

It builds consecutive-season player rows, matches source records, creates shared-profile targets and masks, and normalizes season salary by the salary cap. Salary stays out of model inputs and PCA similarity. The test split is created and preserved here, not scored.

## 1. Packages and project paths

In [ ]:
from pathlib import Path
import unicodedata
import pandas as pd
import json
import hashlib
import numpy as np
from sklearn.model_selection import train_test_split

rootPath = Path.cwd()
while not (rootPath / 'data' / 'raw').exists() and rootPath != rootPath.parent:
    rootPath = rootPath.parent
if not (rootPath / 'data' / 'raw').exists(): raise FileNotFoundError('Open the notebook from inside the project.')
ROOT = rootPath
RAW = ROOT / 'data' / 'raw'
PROCESSED = ROOT / 'data' / 'processed'
processed_folder = PROCESSED
CAP_PATH = RAW / 'salary_cap_history.csv'
PAIRS_PATH = PROCESSED / 'prediction_pairs_rates.csv'
OUTPUT_PATH = PROCESSED / 'salary_cap_normalized_pairs.csv'


## Implementation: `player_matching`

In [ ]:
name_aliases = {
    "Isaac Austin": "Ike Austin",
    "Adama-Alpha Bal": "Adama Bal",
    "Walter Clayton": "Walter Clayton Jr.",
    "Egor Dёmin": "Egor Dëmin",
    "Isaac Fontaine": "Ike Fontaine",
    "Kiwane Lemorris Garris": "Kiwane Garris",
    "DaRon Holmes": "DaRon Holmes II",
    "Eugene Jeter": "Pooh Jeter",
    "Ibo Kutluay": "Ibrahim Kutluay",
    "Rich Manning": "Richard Manning",
    "Stanislav Medvedenko": "Slava Medvedenko",
    "Nathan Mensah": "Nate Mensah",
    "Ronald Murray": "Flip Murray",
    "Norm Richardson": "Norman Richardson",
    "Danny Schayes": "Dan Schayes",
    "Ha Seung-Jin": "Ha Ha",
    "Mike Sweetney": "Michael Sweetney",
    "Clarence Weatherspoon": "Clar. Weatherspoon",
    "Sun Yue": "Sun Sun",
    "Cui Yongxi": "Cui Cui",
    "Wade Baldwin": "Wade Baldwin IV",
    "Brandon Boston Jr.": "Brandon Boston",
    "Brian Bowen": "Brian Bowen II",
    "Reggie Bullock": "Reggie Bullock Jr.",
    "Jimmy Butler": "Jimmy Butler III",
    "John Butler": "John Butler Jr.",
    "Mitch Creek": "Mitchell Creek",
    "Jeff Dowtin": "Jeff Dowtin Jr.",
    "Vince Edwards": "Vincent Edwards",
    "Vítor Luiz Faverani": "Vitor Faverani",
    "Melvin Frazier": "Melvin Frazier Jr.",
    "Michael Frazier": "Michael Frazier II",
    "Harry Giles": "Harry Giles III",
    "Nigel Hayes": "Nigel Hayes-Davis",
    "Ron Holland": "Ronald Holland II",
    "Vince Hunter": "Vincent Hunter",
    "Matthew Hurt": "Matt Hurt",
    "GG Jackson II": "GG Jackson",
    "Trey Jemison": "Trey Jemison III",
    "Perry Jones": "Perry Jones III",
    "Kevin Knox": "Kevin Knox II",
    "Roger Mason": "Roger Mason Jr.",
    "Marcus Morris": "Marcus Morris Sr.",
    "Zach Norvell": "Zach Norvell Jr.",
    "Johnny O'Bryant": "Johnny O'Bryant III",
    "Bobby Portis": "Bobby Portis Jr.",
    "Cameron Reynolds": "Cam Reynolds",
    "Glen Rice Jr.": "Glen Rice",
    "Jermaine Samuels": "Jermaine Samuels Jr.",
    "Tre Scott": "Trevon Scott",
    "Jeff Taylor": "Jeffery Taylor",
    "Xavier Tillman Sr.": "Xavier Tillman",
    "Derrick Walton": "Derrick Walton Jr.",
    "Andrew White": "Andrew White III",
    "Jeenathan Williams": "Nate Williams",
    "Matt Williams": "Matt Williams Jr.",
    "Robert Williams": "Robert Williams III"
}

In [ ]:
explicit_ids = {
    "mitchto02": "203502",
    "mitchto03": "203183",
    "johnsch03": "202419",
    "johnsch04": "203187",
    "willima03": "200766",
    "willima04": "201173",
    "jonesch01": "279",
    "jonesch03": "1869",
    "smithst01": "120"
}

### Implementation: `keep_combined_seasons`

In [ ]:
def keep_combined_seasons(dataframe):
    kept_rows = []

    for key, group in dataframe.groupby(["player_id", "season"]):
        if len(group) == 1:
            kept_rows.append(group)
        else:
            combined = group[group["team"].str.fullmatch(r"TOT|[2-9][0-9]*TM")]

            if len(combined) != 1:
                raise ValueError("Expected one combined record for " + str(key))

            kept_rows.append(combined)

    return pd.concat(kept_rows, ignore_index=True)

### Implementation: `normalize_name`

In [ ]:
def normalize_name(name):
    name = name.casefold()
    name = name.replace("ı", "i")
    name = unicodedata.normalize("NFKD", name)
    name = name.encode("ascii", "ignore").decode("ascii")

    cleaned_name = ""

    for character in name:
        if character.isalnum():
            cleaned_name = cleaned_name + character

    return cleaned_name

### Implementation: `name_key`

In [ ]:
def name_key(name):
    name = name_aliases.get(name, name)
    return normalize_name(name)

## Implementation: `prepare_data`

In [ ]:
FIRST_SEASON = 1997

In [ ]:
LAST_SEASON = 2026

### Implementation: `load_statistics`

In [ ]:
def load_statistics():
    advanced = pd.read_csv(RAW / "basketball_reference" / "Advanced.csv")
    traditional = pd.read_csv(RAW / "basketball_reference" / "Player Per Game.csv")
    advanced = advanced[(advanced["lg"] == "NBA") & advanced["season"].between(FIRST_SEASON, LAST_SEASON)].copy()
    traditional = traditional[(traditional["lg"] == "NBA") & traditional["season"].between(FIRST_SEASON, LAST_SEASON)].copy()
    advanced = keep_combined_seasons(advanced)
    traditional = keep_combined_seasons(traditional)
    columns = ["player_id", "season"]

    for column in traditional.columns:
        if column not in advanced.columns:
            columns.append(column)

    stats = advanced.merge(traditional[columns], on=["player_id", "season"], how="outer", validate="one_to_one", indicator=True)

    if not stats["_merge"].eq("both").all():
        raise ValueError("Traditional and advanced records do not match.")

    stats = stats.drop(columns="_merge")
    frames = []
    nba_columns = ["player_id", "player_name", "season", "age", "gp", "pie", "off_rating", "def_rating", "net_rating", "pace"]

    for year in range(FIRST_SEASON, LAST_SEASON + 1):
        path = RAW / "sportsdataverse" / ("player_season_stats_" + str(year) + ".parquet")
        season = pd.read_parquet(path)
        season = season[(season["season_type"] == "regular-season") & (season["measure_type"] == "advanced") & (season["per_mode"] == "pergame")].copy()

        if len(season) == 0 or not season["season"].eq(year).all():
            raise ValueError("Missing or mislabeled NBA season: " + str(year))

        frames.append(season[nba_columns])

    nba = pd.concat(frames, ignore_index=True)
    nba = nba.rename(columns={"player_id": "nba_player_id", "player_name": "nba_player_name", "age": "nba_age", "gp": "nba_games"})
    nba["nba_player_id"] = nba["nba_player_id"].astype("string")

    if nba.duplicated(["nba_player_id", "season"]).any():
        raise ValueError("Duplicate NBA player-seasons after filtering measures and modes.")

    stats["name_key"] = stats["player"].map(name_key)
    nba["name_key"] = nba["nba_player_name"].map(name_key)
    return stats, nba

### Implementation: `match_players`

In [ ]:
def match_players(stats, nba):
    keys = ["name_key", "season"]
    stats_unique = stats[~stats.duplicated(keys, keep=False)].copy()
    nba_unique = nba[~nba.duplicated(keys, keep=False)].copy()
    named = stats_unique.merge(nba_unique[keys + ["nba_player_id"]], on=keys, how="inner", validate="one_to_one")
    crosswalk = named[["player_id", "nba_player_id"]].drop_duplicates()
    manual = []

    for bbref_id, nba_id in explicit_ids.items():
        manual.append({"player_id": bbref_id, "nba_player_id": nba_id})

    crosswalk = pd.concat([crosswalk, pd.DataFrame(manual)], ignore_index=True).drop_duplicates()

    if crosswalk["player_id"].duplicated().any() or crosswalk["nba_player_id"].duplicated().any():
        raise ValueError("Inconsistent player ID mappings need review.")

    matched = stats.merge(crosswalk, on="player_id", how="left", validate="many_to_one")
    matched = matched.merge(nba.drop(columns="name_key"), on=["nba_player_id", "season"], how="left", validate="one_to_one", indicator=True)
    unmatched = matched[matched["_merge"] != "both"].copy()
    unmatched.to_csv(PROCESSED / "unmatched_players.csv", index=False)

    if len(unmatched) > 0:
        print(unmatched[["player", "player_id", "season"]].to_string(index=False))
        raise ValueError("Some players require an alias or explicit ID mapping.")

    matched = matched.drop(columns="_merge")

    if len(matched) != len(nba) or matched.duplicated(["nba_player_id", "season"]).any():
        raise ValueError("The two sources do not cover the same unique player-seasons.")

    matched["age_difference"] = (matched["age"] - matched["nba_age"]).abs()
    review = matched[(matched["age_difference"] > 1) | matched["age_difference"].isna() | matched["g"].ne(matched["nba_games"])].copy()
    review.to_csv(PROCESSED / "player_source_review.csv", index=False)
    crosswalk.to_csv(PROCESSED / "player_id_crosswalk.csv", index=False)
    return matched, review

### Implementation: `build_pairs`

In [ ]:
def build_pairs(matched):
    targets = matched[["nba_player_id", "season", "pie", "ws", "vorp"]].rename(columns={"season": "target_season", "pie": "next_season_pie", "ws": "next_season_ws", "vorp": "next_season_vorp"})
    targets["season"] = targets["target_season"] - 1
    pairs = matched.merge(targets, on=["nba_player_id", "season"], how="inner", validate="one_to_one")
    missing = pairs["pie"].isna() | pairs["next_season_pie"].isna()
    pairs[missing].to_csv(PROCESSED / "pairs_missing_pie.csv", index=False)
    pairs = pairs[~missing].copy()
    pairs["target_season"] = pairs["target_season"].astype(int)
    return pairs

### Implementation: `attach_salaries`

In [ ]:
def attach_salaries(pairs, matched):
    # Preserve all previously reviewed salary amounts as the first choice.
    salaries = pd.read_csv(RAW / "salaries" / "BargainBall_listed_salaries.csv", dtype={"player_id": "string"})
    columns = ["player_id", "target_season", "target_salary", "source_url", "salary_needs_source_review"]
    salaries = salaries[columns].rename(columns={"source_url": "salary_source_url"})
    salaries["salary_method"] = "previous_listed_lookup"
    frames = []

    for year in range(2010, LAST_SEASON + 1):
        path = RAW / "bbref_salaries" / ("bbref_salaries_" + str(year) + ".parquet")
        frame = pd.read_parquet(path)

        if not frame["season"].eq(year).all() or frame.duplicated(["season", "team_id", "bbref_id"]).any():
            raise ValueError("Unexpected salary archive records: " + str(year))

        frames.append(frame)

    team_salaries = pd.concat(frames, ignore_index=True)
    team_salaries.to_csv(PROCESSED / "salary_team_records.csv", index=False)
    archive_rows = []

    for key, group in team_salaries.groupby(["bbref_id", "season"]):
        if group["salary"].isna().any():
            continue

        amount = group["salary"].sum()

        if amount > 0:
            archive_rows.append({"player_id": key[0], "target_season": int(key[1]), "target_salary": amount, "salary_source_url": "https://basketball-data.billmill.org/nba/bbref/bbref_salaries/season=" + str(key[1]) + "/data.parquet", "salary_needs_source_review": False, "salary_method": "bbref_team_salary_sum"})

    archive = pd.DataFrame(archive_rows)
    salaries = pd.concat([salaries, archive], ignore_index=True).drop_duplicates(["player_id", "target_season"], keep="first")

    # Historical salaries label seasons by their STARTING year.
    historical = pd.read_csv(RAW / "salaries" / "historical_salaries.csv").drop_duplicates()
    historical["target_season"] = historical["seasonStartYear"].astype(int) + 1
    historical["name_key"] = historical["playerName"].map(name_key)
    historical["target_salary"] = pd.to_numeric(historical["salary"].str.replace("$", "", regex=False).str.replace(",", "", regex=False), errors="raise")
    names = matched[["player_id", "season", "name_key"]].rename(columns={"season": "target_season"})
    nba_names = matched[["player_id", "season", "nba_player_name"]].rename(columns={"season": "target_season"})
    nba_names["name_key"] = nba_names["nba_player_name"].map(name_key)
    names = pd.concat([names, nba_names.drop(columns="nba_player_name")], ignore_index=True).drop_duplicates()
    names = names[~names.duplicated(["name_key", "target_season"], keep=False)]
    historical = historical.merge(names, on=["name_key", "target_season"], how="inner", validate="many_to_one")
    historical = historical[["player_id", "target_season", "target_salary"]].drop_duplicates()
    ambiguous = historical.duplicated(["player_id", "target_season"], keep=False)
    historical[ambiguous].to_csv(PROCESSED / "historical_salary_conflicts.csv", index=False)
    historical = historical[~ambiguous & historical["target_salary"].gt(0)].copy()
    historical["salary_source_url"] = "https://github.com/logan-lauton/NBA-Player-HUD/blob/main/NBA%20Salaries(1990-2023).csv"
    historical["salary_needs_source_review"] = True
    historical["salary_method"] = "historical_listed_salary"
    salaries = pd.concat([salaries, historical], ignore_index=True).drop_duplicates(["player_id", "target_season"], keep="first")

    # Another archive contains Basketball-Reference IDs, avoiding name guesses.
    older = pd.read_csv(RAW / "salaries" / "bbref_historical_salaries.csv")
    older["player_id"] = older["url"].str.extract(r"/players/[a-z]/([^/.]+)")
    older["target_season"] = older["season"].str[:4].astype(int) + 1
    older["amount"] = pd.to_numeric(older["salary"].str.replace("$", "", regex=False).str.replace(",", "", regex=False), errors="coerce")
    older = older.drop_duplicates(["team_url", "season", "player_id", "amount"])
    older_rows = []

    for key, group in older.groupby(["player_id", "target_season"]):
        if group["amount"].isna().any() or group["team_url"].duplicated().any():
            continue

        amount = group["amount"].sum()

        if amount > 0:
            older_rows.append({"player_id": key[0], "target_season": int(key[1]), "target_salary": amount, "salary_source_url": "https://github.com/ramvibhakar/basketball-analytics/blob/master/data/player_salary.csv", "salary_needs_source_review": True, "salary_method": "historical_bbref_team_sum"})

    salaries = pd.concat([salaries, pd.DataFrame(older_rows)], ignore_index=True).drop_duplicates(["player_id", "target_season"], keep="first")

    # This archive's season is the ENDING year (e.g. Shaq's 1999-2000 salary).
    alternative = pd.read_csv(RAW / "salaries" / "alternative_salaries.csv")
    alternative["name_key"] = alternative["name"].map(name_key)
    alternative = alternative.rename(columns={"season": "target_season", "salary": "target_salary"})
    alternative = alternative.merge(names, on=["name_key", "target_season"], how="inner", validate="many_to_one")
    alternative = alternative[["player_id", "target_season", "target_salary"]].drop_duplicates()
    alternative = alternative[~alternative.duplicated(["player_id", "target_season"], keep=False) & alternative["target_salary"].gt(0)].copy()
    alternative["salary_source_url"] = "https://github.com/erikgregorywebb/datasets/blob/master/nba-salaries.csv"
    alternative["salary_needs_source_review"] = True
    alternative["salary_method"] = "alternative_listed_salary"
    salaries = pd.concat([salaries, alternative], ignore_index=True).drop_duplicates(["player_id", "target_season"], keep="first")

    # HoopsHype snapshot mirrored by MrBridge; retain its published amounts.
    snapshot = pd.read_csv(RAW / "salaries" / "hoopshype_snapshot.csv").drop_duplicates()
    salary_aliases = {"Patrick Baldwin": "Patrick Baldwin Jr.", "Wendell Moore": "Wendell Moore Jr.", "TyTy Washington": "TyTy Washington Jr.", "Kevin McCullar": "Kevin McCullar Jr."}
    snapshot["name_key"] = snapshot["player"].replace(salary_aliases).map(name_key)
    snapshot["target_season"] = snapshot["season_year"].astype(int) + 1
    snapshot = snapshot[snapshot["target_season"].between(FIRST_SEASON + 1, LAST_SEASON)].copy()
    snapshot = snapshot.merge(names, on=["name_key", "target_season"], how="inner", validate="many_to_one", suffixes=("_hoopshype", ""))
    snapshot.to_csv(PROCESSED / "hoopshype_salary_records.csv", index=False)
    snapshot_rows = []

    for key, group in snapshot.groupby(["player_id", "target_season"]):
        group = group.drop_duplicates(["team_id", "salary"])

        if group["team_id"].duplicated().any() or group["salary"].isna().any() or not group["salary"].gt(0).all():
            continue

        snapshot_rows.append({"player_id": key[0], "target_season": int(key[1]), "target_salary": group["salary"].sum(), "salary_source_url": "https://huggingface.co/datasets/Mr-Bridge/nba-salary-cap-contracts-2016-2026/blob/main/player_salaries.csv", "salary_needs_source_review": True, "salary_method": "hoopshype_snapshot_team_sum"})

    salaries = pd.concat([salaries, pd.DataFrame(snapshot_rows)], ignore_index=True).drop_duplicates(["player_id", "target_season"], keep="first")

    # Resolve the same-name collision in the NBA-Player-HUD historical file:
    # smithch04 is the Clippers player, while the $4,375,143 amount belongs
    # to the separate Spurs player with the same name.
    verified_corrections = pd.DataFrame([
        {
            "player_id": "smithch04",
            "target_season": 1999,
            "target_salary": 677080,
            "salary_source_url": "https://www.hoopshype.com/salaries/players/charles-smith/3176",
            "salary_needs_source_review": False,
            "salary_method": "verified_name_collision_correction",
        },
        {
            "player_id": "bryanma01",
            "target_season": 2001,
            "target_salary": 3029268,
            "salary_source_url": "https://www.hoopshype.com/salaries/players/mark-bryant/42",
            "salary_needs_source_review": False,
            "salary_method": "verified_source_confirmation",
        },
        {
            "player_id": "richaja01",
            "target_season": 2009,
            "target_salary": 12664335,
            "salary_source_url": "https://www.hoopshype.com/salaries/players/jason-richardson/3496",
            "salary_needs_source_review": False,
            "salary_method": "verified_source_confirmation",
        },
    ])
    external_verifications_path = RAW / "salaries" / "verified_salary_overrides.csv"
    if external_verifications_path.exists():
        external_verifications = pd.read_csv(external_verifications_path)
        verified_corrections = pd.concat([verified_corrections, external_verifications], ignore_index=True)
    salaries = pd.concat([salaries, verified_corrections], ignore_index=True)
    salaries = salaries.drop_duplicates(["player_id", "target_season"], keep="last")
    pd.DataFrame([{
        "player_id": "smithch04",
        "player": "Charles Smith",
        "target_season": 1999,
        "previous_salary": 5052223,
        "corrected_salary": 677080,
        "reason": "The historical name match combined the Clippers player's salary with a different Charles Smith's Spurs salary.",
        "verification_sources": "https://www.hoopshype.com/salaries/players/charles-smith/3176; https://www.basketball-reference.com/players/s/smithch04.html",
    }, {
        "player_id": "bryanma01",
        "player": "Mark Bryant",
        "target_season": 2001,
        "previous_salary": 3029268,
        "corrected_salary": 3029268,
        "reason": "HoopsHype shows $2,200,000 with Cleveland and $829,268 with Dallas; their sum confirms the listed season salary.",
        "verification_sources": "https://www.hoopshype.com/salaries/players/mark-bryant/42; https://www.basketball-reference.com/players/b/bryanma01.html",
    }, {
        "player_id": "richaja01",
        "player": "Jason Richardson",
        "target_season": 2009,
        "previous_salary": 12664335,
        "corrected_salary": 12664335,
        "reason": "HoopsHype shows $12,222,221 with Phoenix and $442,114 with Miami; their sum confirms the listed season salary.",
        "verification_sources": "https://www.hoopshype.com/salaries/players/jason-richardson/3496; https://www.basketball-reference.com/players/r/richaja01.html",
    }]).to_csv(PROCESSED / "verified_salary_corrections.csv", index=False)
    salaries.to_csv(PROCESSED / "salary_lookup.csv", index=False)
    pairs = pairs.merge(salaries, on=["player_id", "target_season"], how="left", validate="one_to_one")
    pairs["salary_available"] = pairs["target_salary"].notna() & pairs["target_salary"].gt(0)
    pairs["salary_needs_source_review"] = pairs["salary_needs_source_review"].astype("boolean").fillna(True)
    return pairs

### Implementation: `main`

In [ ]:
def main():
    PROCESSED.mkdir(parents=True, exist_ok=True)
    stats, nba = load_statistics()
    matched, review = match_players(stats, nba)
    pairs = attach_salaries(build_pairs(matched), matched)
    pairs = pairs.sort_values(["player_id", "season"]).reset_index(drop=True)

    if pairs.duplicated(["player_id", "season"]).any() or not pairs["target_season"].eq(pairs["season"] + 1).all():
        raise ValueError("Invalid or duplicate consecutive-season pairs.")

    # Replace the main dataset only after the entire CSV has been written.
    temporary = PROCESSED / "prediction_pairs.csv.part"
    pairs.to_csv(temporary, index=False)
    temporary.replace(PROCESSED / "prediction_pairs.csv")
    pairs[~pairs["salary_available"]].to_csv(PROCESSED / "missing_salaries.csv", index=False)
    pairs[pairs["salary_available"]].to_csv(PROCESSED / "salary_analysis_pairs.csv", index=False)
    coverage = pairs.groupby("target_season").agg(prediction_pairs=("player_id", "size"), pairs_with_salary=("salary_available", "sum"))
    coverage["missing_salary"] = coverage["prediction_pairs"] - coverage["pairs_with_salary"]
    coverage.to_csv(PROCESSED / "coverage_by_season.csv")
    summary = {"first_input_season": int(pairs["season"].min()), "last_input_season": int(pairs["season"].max()), "first_target_season": int(pairs["target_season"].min()), "last_target_season": int(pairs["target_season"].max()), "matched_player_seasons": len(matched), "prediction_pairs": len(pairs), "pairs_with_salary": int(pairs["salary_available"].sum()), "missing_salary": int((~pairs["salary_available"]).sum()), "salaries_needing_review": int(pairs["salary_needs_source_review"].sum()), "source_discrepancies_retained": len(review)}
    summary["salaries_needing_review"] = int((pairs["salary_needs_source_review"] & pairs["salary_available"]).sum())
    (PROCESSED / "dataset_summary.json").write_text(json.dumps(summary, indent=2) + "\n", encoding="utf-8")
    print("Matched player-seasons: " + str(len(matched)))
    print("Prediction pairs: " + str(len(pairs)))
    print("Pairs with salary: " + str(summary["pairs_with_salary"]))
    print("Pairs missing salary (retained for performance training): " + str(summary["missing_salary"]))
    print("Saved dataset to: " + str(PROCESSED / "prediction_pairs.csv"))

runDataPreparation = main


## Implementation: `preprocess`

In [ ]:
numeric_features = [
    "age",
    "g",
    "mp_per_game",
    "pts_per_game",
    "trb_per_game",
    "ast_per_game",
    "stl_per_game",
    "blk_per_game",
    "tov_per_game",
    "ts_percent",
    "usg_percent",
    "per",
    "ws",
    "ws_48",
    "bpm",
    "vorp",
    "pie"
]

In [ ]:
target_columns = {"pie": "next_season_pie", "ws": "next_season_ws", "vorp": "next_season_vorp"}

### Implementation: `main`

In [ ]:
def main():
    dataset_path = processed_folder / "prediction_pairs.csv"
    data = pd.read_csv(dataset_path)
    summary = json.loads((processed_folder / "dataset_summary.json").read_text(encoding="utf-8"))

    if len(data) != summary["prediction_pairs"]:
        raise ValueError("Dataset row count differs from the preparation summary. Rerun the player-pair preparation section.")

    for target, column in target_columns.items():
        if column not in data.columns:
            raise ValueError("Missing " + column + ". Rerun the player-pair preparation section.")

        if not np.isfinite(data[column].to_numpy(dtype=float)).all():
            raise ValueError("Some " + target + " targets are missing or infinite. Do not impute targets.")

    if data.duplicated(["player_id", "season"]).any():
        raise ValueError("Duplicate prediction examples.")

    if not data["target_season"].eq(data["season"] + 1).all():
        raise ValueError("Targets must come from the immediately following season.")

    # Randomly split the rows: 70% training, 15% development, 15% test.
    train_data, remaining_data = train_test_split(data, test_size=0.30, random_state=42)
    dev_data, test_data = train_test_split(remaining_data, test_size=0.50, random_state=42)
    datasets = {"train": train_data.copy(), "dev": dev_data.copy(), "test": test_data.copy()}

    # Older seasons also have a few missing PER, usage, WS/48, and BPM values.
    # Learn every replacement value using training data only.
    medians = {}

    for column in numeric_features:
        median = datasets["train"][column].median()

        if not np.isfinite(median):
            raise ValueError("No finite training median for " + column)

        medians[column] = float(median)

    inputs = {}
    targets = {}

    for split_name, rows in datasets.items():
        for column in numeric_features:
            rows[column] = rows[column].fillna(medians[column])

        inputs[split_name] = rows[numeric_features + ["pos"]].copy()
        targets[split_name] = rows["next_season_pie"].copy()

    # Standardize within each input season using training statistics only.
    training_means = {}
    training_stds = {}

    for column in numeric_features:
        season_means = datasets["train"].groupby("season")[column].mean()
        season_stds = datasets["train"].groupby("season")[column].std(ddof=0)
        season_stds = season_stds.replace(0, 1)
        training_means[column] = season_means.to_dict()
        training_stds[column] = season_stds.to_dict()

        for split_name, rows in datasets.items():
            means = rows["season"].map(season_means)
            stds = rows["season"].map(season_stds)
            inputs[split_name][column] = (rows[column] - means) / stds

    # Learn positions from training data and use the same columns in every split.
    positions = sorted(datasets["train"]["pos"].unique())

    for split_name, rows in datasets.items():
        if not rows["pos"].isin(positions).all():
            raise ValueError("An unknown position appeared in the " + split_name + " split.")

        for position in positions:
            column_name = "pos_" + position
            inputs[split_name][column_name] = (rows["pos"] == position).astype(int)

        inputs[split_name] = inputs[split_name].drop(columns=["pos"])

    # Use original PIE values to evaluate the previous-season PIE baseline.
    baseline_errors = datasets["dev"]["pie"].to_numpy() - targets["dev"].to_numpy()
    baseline_rmse = np.sqrt(np.mean(baseline_errors ** 2))
    baseline_mae = np.mean(np.abs(baseline_errors))

    # Save arrays and corresponding player rows in the same order.
    model_data = {}

    for split_name, rows in datasets.items():
        model_inputs = inputs[split_name].to_numpy(dtype=np.float32)
        model_targets = targets[split_name].to_numpy(dtype=np.float32)

        if not np.isfinite(model_inputs).all() or not np.isfinite(model_targets).all():
            raise ValueError("Missing or infinite model values in the " + split_name + " split.")

        model_data["X_" + split_name] = model_inputs
        model_data["y_" + split_name] = model_targets

        for target, column in target_columns.items():
            # All target arrays share exactly the same input rows and row order.
            model_data["y_" + target + "_" + split_name] = rows[column].to_numpy(dtype=np.float32)
            model_data["baseline_" + target + "_" + split_name] = rows[target].to_numpy(dtype=np.float32)

        rows.to_csv(processed_folder / (split_name + "_rows.csv"), index=False)

    model_data["baseline_dev"] = datasets["dev"]["pie"].to_numpy(dtype=np.float32)
    model_data["baseline_test"] = datasets["test"]["pie"].to_numpy(dtype=np.float32)
    model_data["feature_names"] = inputs["train"].columns.to_numpy(dtype=str)
    model_data["target_names"] = np.array(list(target_columns), dtype=str)

    output_path = processed_folder / "model_data.npz"
    np.savez_compressed(output_path, **model_data)

    # Preserve the learned transformations for reuse.
    preprocessing = {
        "dataset_sha256": hashlib.sha256(dataset_path.read_bytes()).hexdigest(),
        "examples": len(data),
        "random_state": 42,
        "split_method": "random prediction rows; 70% train, 15% dev, 15% test",
        "numeric_features": numeric_features,
        "target_columns": target_columns,
        "positions": positions,
        "feature_names": list(inputs["train"].columns),
        "medians": medians,
        "season_means": training_means,
        "season_stds": training_stds
    }

    settings_path = processed_folder / "preprocessing.json"
    settings_path.write_text(json.dumps(preprocessing, indent=2, allow_nan=False) + "\n", encoding="utf-8")

    # Print one concise summary.
    print("Examples: " + str(len(data)))

    for split_name in datasets:
        print(split_name + " input shape: " + str(inputs[split_name].shape))

    for target, column in target_columns.items():
        errors = datasets["dev"][target].to_numpy() - datasets["dev"][column].to_numpy()
        print(target.upper() + " development baseline RMSE: " + str(float(np.sqrt(np.mean(errors ** 2)))))
        print(target.upper() + " development baseline MAE: " + str(float(np.mean(np.abs(errors)))))
    print("Saved model data to: " + str(output_path))

runBasePreprocessing = main


## Implementation: `profile_targets`

In [ ]:
SKILL_TARGETS = [
    "pie",
    "ws_48",
    "bpm",
    "ts_percent",
    "usg_percent",
    "trb_percent",
    "ast_percent",
    "stl_percent",
    "blk_percent",
    "tov_percent",
    "x3p_ar",
    "f_tr",
    "e_fg_percent",
    "x3p_percent",
    "ft_percent",
]

In [ ]:
PROFILE_TARGETS = SKILL_TARGETS + ["minutes"]

In [ ]:
PCA_TARGETS = [
    target for target in SKILL_TARGETS
    if target not in {"x3p_percent", "ft_percent"}
]

In [ ]:
NEXT_SEASON_COLUMNS = {
    target: ("next_season_minutes" if target == "minutes" else "next_season_" + target)
    for target in PROFILE_TARGETS
}

In [ ]:
BASELINE_COLUMNS = {
    target: ("mp" if target == "minutes" else target)
    for target in PROFILE_TARGETS
}

## Implementation: `prepare_rates`

### Implementation: `main`

In [ ]:
def main():
    processed = ROOT / "data" / "processed"
    pairs = pd.read_csv(processed / "prediction_pairs.csv")
    season_range = (pairs["target_season"].min(), pairs["target_season"].max())
    advanced = pd.read_csv(ROOT / "data" / "raw" / "basketball_reference" / "Advanced.csv")
    advanced = advanced[
        (advanced["lg"] == "NBA")
        & advanced["season"].between(*season_range)
    ].copy()
    advanced = keep_combined_seasons(advanced)
    advanced_fields = [
        "player_id", "season", "ws_48", "bpm", "mp", "g", "ws", "vorp",
        "ts_percent", "usg_percent", "trb_percent", "ast_percent",
        "stl_percent", "blk_percent", "tov_percent", "x3p_ar", "f_tr",
    ]
    advanced_targets = advanced[advanced_fields].rename(
        columns={
            "season": "target_season",
            "ws_48": "next_season_ws_48",
            "bpm": "next_season_bpm",
            "mp": "next_season_minutes",
            "g": "next_season_games",
            "ws": "source_next_ws",
            "vorp": "source_next_vorp",
            **{name: "next_season_" + name for name in PCA_TARGETS
               if name in advanced_fields},
        }
    )
    extended = pairs.merge(
        advanced_targets,
        on=["player_id", "target_season"],
        how="left",
        sort=False,
        validate="one_to_one",
        indicator=True,
    )
    if len(extended) != len(pairs) or not extended["_merge"].eq("both").all():
        raise ValueError("Some target-season records did not match Advanced.csv.")

    per_game = pd.read_csv(
        ROOT / "data" / "raw" / "basketball_reference" / "Player Per Game.csv"
    )
    per_game = per_game[
        (per_game["lg"] == "NBA")
        & per_game["season"].between(*season_range)
    ].copy()
    per_game = keep_combined_seasons(per_game)
    per_game_fields = [
        "player_id", "season", "e_fg_percent", "x3p_percent", "ft_percent",
        "mp_per_game",
    ]
    per_game_targets = per_game[per_game_fields].rename(
        columns={
            "season": "target_season",
            "e_fg_percent": "next_season_e_fg_percent",
            "x3p_percent": "next_season_x3p_percent",
            "ft_percent": "next_season_ft_percent",
            "mp_per_game": "next_season_mp_per_game",
        }
    )
    extended = extended.merge(
        per_game_targets,
        on=["player_id", "target_season"],
        how="left",
        sort=False,
        validate="one_to_one",
        indicator="per_game_merge",
    )
    if not extended["per_game_merge"].eq("both").all():
        raise ValueError("Some target-season records did not match Player Per Game.csv.")
    extended = extended.drop(columns=["per_game_merge"])

    pd.testing.assert_frame_equal(extended[["player_id", "season"]], pairs[["player_id", "season"]])

    for target in ["ws", "vorp"]:
        np.testing.assert_allclose(extended["source_next_" + target], extended["next_season_" + target], rtol=0, atol=0)

    extended = extended.drop(columns=["_merge", "source_next_ws", "source_next_vorp"])
    extended["rate_valid"] = True

    # Never impute an undefined target or a missing baseline for comparison.
    for column in ["next_season_pie", "next_season_ws_48", "next_season_bpm", "pie", "ws_48", "bpm"]:
        extended["rate_valid"] = extended["rate_valid"] & np.isfinite(extended[column].to_numpy(dtype=float))

    profile_cols = [NEXT_SEASON_COLUMNS[target] for target in PCA_TARGETS]
    extended["skill_profile_valid"] = np.isfinite(
        extended[profile_cols].to_numpy(dtype=float)
    ).all(axis=1)
    extended["skill_profile_valid"] = (
        extended["skill_profile_valid"]
        & extended["next_season_minutes"].ge(0)
    )

    path = processed / "prediction_pairs_rates.csv"
    temporary = path.with_suffix(".csv.tmp")
    extended.to_csv(temporary, index=False)
    temporary.replace(path)
    extended[~extended["rate_valid"]].to_csv(processed / "rate_excluded_rows.csv", index=False)
    print("Original examples retained: " + str(len(extended)))
    print("Examples with defined rate targets and baselines: " + str(int(extended["rate_valid"].sum())))
    print("Undefined rate rows (retained in original experiment): " + str(int((~extended["rate_valid"]).sum())))
    print("Complete next-season skill profiles for PCA: " + str(int(extended["skill_profile_valid"].sum())))
    print("Per-target missing labels:")
    print(extended[[NEXT_SEASON_COLUMNS[target] for target in NEXT_SEASON_COLUMNS]].isna().sum().to_string())
    print("Saved to: " + str(path))

attachProfileTargets = main


## Implementation: `preprocess_rates`

### Implementation: `main`

In [ ]:
def main():
    folder = ROOT / "data" / "processed"
    base_path = folder / "model_data.npz"
    pairs = pd.read_csv(folder / "prediction_pairs_rates.csv")
    pairs = pairs.set_index(["player_id", "season"], verify_integrity=True)
    arrays = {}
    sizes = {}

    with np.load(base_path, allow_pickle=False) as original:
        for key in original.files:
            arrays[key] = original[key].copy()

    for split in ["train", "dev", "test"]:
        rows = pd.read_csv(folder / (split + "_rows.csv"))
        keys = pd.MultiIndex.from_frame(rows[["player_id", "season"]])
        aligned = pairs.loc[keys].reset_index()

        if len(rows) != arrays["X_" + split].shape[0]:
            raise ValueError("Split metadata and model inputs have different row counts.")

        pd.testing.assert_frame_equal(rows[["player_id", "season"]], aligned[["player_id", "season"]])
        np.testing.assert_array_equal(arrays["y_pie_" + split], aligned["next_season_pie"].to_numpy(dtype=np.float32))

        for target in ["ws_48", "bpm"]:
            arrays["y_" + target + "_" + split] = aligned["next_season_" + target].to_numpy(dtype=np.float32)
            arrays["baseline_" + target + "_" + split] = aligned[target].to_numpy(dtype=np.float32)

        profile_y = aligned[
            [NEXT_SEASON_COLUMNS[target] for target in PROFILE_TARGETS]
        ].to_numpy(dtype=np.float64)
        profile_baseline = aligned[
            [BASELINE_COLUMNS[target] for target in PROFILE_TARGETS]
        ].to_numpy(dtype=np.float64)
        arrays["profile_y_" + split] = profile_y
        arrays["profile_mask_" + split] = np.isfinite(profile_y)
        arrays["profile_baseline_" + split] = profile_baseline
        arrays["profile_baseline_mask_" + split] = np.isfinite(profile_baseline)
        arrays["skill_profile_mask_" + split] = (
            aligned["skill_profile_valid"].to_numpy(dtype=bool)
        )

        mask = aligned["rate_valid"].to_numpy(dtype=bool)
        arrays["rate_mask_" + split] = mask
        arrays["next_season_minutes_" + split] = aligned["next_season_minutes"].to_numpy(dtype=np.float64)
        arrays["prior_minutes_" + split] = aligned["mp"].to_numpy(dtype=np.float64)
        path = folder / (split + "_rows_rates.csv")
        temporary = path.with_suffix(".csv.tmp")
        aligned.to_csv(temporary, index=False)
        temporary.replace(path)
        sizes[split] = {"original": len(rows), "valid_rate_examples": int(mask.sum()), "excluded": int((~mask).sum())}
        print(split + ": " + str(int(mask.sum())) + " valid rate examples out of " + str(len(rows)))

    arrays["base_model_data_sha256"] = np.array(hashlib.sha256(base_path.read_bytes()).hexdigest())
    arrays["profile_target_names"] = np.asarray(PROFILE_TARGETS, dtype=str)
    arrays["pca_target_names"] = np.asarray(PCA_TARGETS, dtype=str)
    path = folder / "model_data_rates.npz"
    np.savez_compressed(path, **arrays)
    report = {
        "base_model_data_sha256": str(arrays["base_model_data_sha256"]),
        "splits": sizes,
        "input_arrays": "copied unchanged from model_data.npz",
        "profile_targets": PROFILE_TARGETS,
        "pca_skill_targets": PCA_TARGETS,
        "missing_target_policy": "Per-target masks; no label imputation.",
        "next_season_minutes": "Predicted by the shared model; kept outside the skill-profile PCA.",
        "test_metrics_computed": False,
    }
    (folder / "rate_preprocessing.json").write_text(json.dumps(report, indent=2) + "\n", encoding="utf-8")
    print("Saved new arrays to: " + str(path))

prepareProfileArrays = main


## Implementation: `analyze_salary_cap`

### Implementation: `main`

In [ ]:
def main():
    caps = pd.read_csv(CAP_PATH)
    pairs = pd.read_csv(PAIRS_PATH)
    caps["target_season"] = pd.to_numeric(caps["target_season"], errors="raise").astype(int)
    caps["salary_cap_usd"] = pd.to_numeric(caps["salary_cap_usd"], errors="raise")
    pairs["target_season"] = pd.to_numeric(pairs["target_season"], errors="raise").astype(int)
    pairs["target_salary"] = pd.to_numeric(pairs["target_salary"], errors="coerce")

    if(caps["target_season"].duplicated().any()):
        raise ValueError("Salary-cap source has duplicate target-season rows.")
    if(caps["salary_cap_usd"].isna().any() or (caps["salary_cap_usd"] <= 0).any()):
        raise ValueError("Every salary-cap value must be finite and positive.")
    if(pairs.duplicated(["player_id", "target_season"]).any()):
        raise ValueError("Player-season pairs must be unique before cap join.")

    expected_seasons = list(range(int(pairs["target_season"].min()), int(pairs["target_season"].max()) + 1))
    if(sorted(caps["target_season"].tolist()) != expected_seasons):
        raise ValueError("Cap table must contain each target season exactly once, without gaps.")

    cap_seasons = caps[["target_season", "season", "salary_cap_usd", "source", "source_url", "accessed_date"]].rename(columns={"season": "salary_cap_season"})
    normalized = pairs.merge(cap_seasons, on="target_season", how="left", validate="many_to_one", indicator=True)

    if(not normalized["_merge"].eq("both").all()):
        raise ValueError("Some target seasons did not match a salary-cap value.")

    normalized = normalized.drop(columns=["_merge"])
    normalized["salary_cap_share"] = normalized["target_salary"] / normalized["salary_cap_usd"]
    normalized["salary_cap_share_pct"] = 100.0 * normalized["salary_cap_share"]
    normalized["salary_available_check"] = normalized["target_salary"].notna() & normalized["target_salary"].gt(0)
    recorded_available = normalized["salary_available"].astype("boolean").fillna(False)

    if(not normalized["salary_available_check"].eq(recorded_available).all()):
        raise ValueError("Computed salary availability does not match the saved availability flag.")

    normalized["salary_needs_source_review"] = normalized["salary_needs_source_review"].astype("boolean").fillna(True)
    normalized["salary_reviewed_and_available"] = normalized["salary_available_check"] & ~normalized["salary_needs_source_review"]
    normalized["source_review_available_salary"] = normalized["salary_available_check"] & normalized["salary_needs_source_review"]
    normalized["source_review_missing_salary"] = ~normalized["salary_available_check"] & normalized["salary_needs_source_review"]
    normalized["salary_cap_share_over_100pct"] = normalized["salary_cap_share"].gt(1.0)
    normalized.to_csv(OUTPUT_PATH, index=False)

    coverage = normalized.groupby(["target_season", "salary_cap_season"], sort=True).agg(
        prediction_pairs=("player_id", "size"),
        salary_available=("salary_available_check", "sum"),
        salary_missing=("salary_available_check", lambda values: int((~values).sum())),
        available_salary_source_review=("source_review_available_salary", "sum"),
        missing_salary_flagged_for_review=("source_review_missing_salary", "sum"),
        reviewed_salary_available=("salary_reviewed_and_available", "sum"),
        cap_share_over_100pct=("salary_cap_share_over_100pct", "sum"),
        cap_share_median_pct=("salary_cap_share_pct", "median"),
        cap_share_mean_pct=("salary_cap_share_pct", "mean"),
    ).reset_index()
    coverage.to_csv(PROCESSED / "salary_cap_coverage_by_season.csv", index=False)

    source_review_audit = normalized.groupby(["salary_method", "salary_available_check", "salary_needs_source_review"], dropna=False).agg(
        records=("player_id", "size"),
        cap_share_median_pct=("salary_cap_share_pct", "median"),
        cap_share_over_100pct=("salary_cap_share_over_100pct", "sum"),
    ).reset_index()
    source_review_audit.to_csv(PROCESSED / "salary_cap_source_review_audit.csv", index=False)

    high_shares = normalized[normalized["salary_cap_share_over_100pct"]].copy()
    high_shares.to_csv(PROCESSED / "salary_cap_shares_over_100pct.csv", index=False)

    available = normalized[normalized["salary_available_check"]]
    review_rows = available[available["salary_needs_source_review"]]
    reviewed_rows = available[~available["salary_needs_source_review"]]
    summary = {
        "cap_source": "Basketball-Reference NBA Salary Cap History",
        "cap_source_url": "https://www.basketball-reference.com/contracts/salary-cap-history.html",
        "cap_source_accessed_date": "2026-11-21",
        "season_key": "target_season is the season ending year; e.g. target_season 2026 maps to 2025-26",
        "normalization": "salary_cap_share = target_salary / salary_cap_usd; percentage is 100 * salary_cap_share",
        "prediction_pairs": int(len(normalized)),
        "first_target_season": int(normalized["target_season"].min()),
        "last_target_season": int(normalized["target_season"].max()),
        "cap_seasons": int(caps["target_season"].nunique()),
        "rows_missing_cap": int(normalized["salary_cap_usd"].isna().sum()),
        "salary_available": int(normalized["salary_available_check"].sum()),
        "salary_missing_preserved": int((~normalized["salary_available_check"]).sum()),
        "available_salary_rows_flagged_for_source_review": int(len(review_rows)),
        "available_salary_rows_not_flagged_for_source_review": int(len(reviewed_rows)),
        "available_salary_cap_shares_over_100pct": int(len(high_shares)),
        "over_100pct_interpretation": "This is a descriptive ratio, not an invalid-record rule: NBA is a soft-cap league and an individual salary may exceed the team cap.",
        "salary_missing_rows_removed_or_imputed": 0,
        "source_review_rows_removed": 0,
        "output_csv": str(OUTPUT_PATH.relative_to(ROOT)),
        "coverage_csv": str((PROCESSED / "salary_cap_coverage_by_season.csv").relative_to(ROOT)),
        "source_review_audit_csv": str((PROCESSED / "salary_cap_source_review_audit.csv").relative_to(ROOT)),
        "over_100pct_examples_csv": str((PROCESSED / "salary_cap_shares_over_100pct.csv").relative_to(ROOT)),
    }
    (PROCESSED / "salary_cap_normalization_summary.json").write_text(json.dumps(summary, indent=2) + "\n", encoding="utf-8")

    print("Salary-cap normalization complete.")
    print("Target seasons: " + str(summary["first_target_season"]) + "-" + str(summary["last_target_season"]) + " (" + str(summary["cap_seasons"]) + " cap values)")
    print("Salary rows: " + str(summary["salary_available"]) + " available; " + str(summary["salary_missing_preserved"]) + " missing preserved")
    print("Available salaries flagged for source review: " + str(summary["available_salary_rows_flagged_for_source_review"]))
    print("Salary/cap shares over 100% (valid in a soft-cap league): " + str(summary["available_salary_cap_shares_over_100pct"]))
    print("Wrote: " + str(OUTPUT_PATH))

runSalaryCapNormalization = main


## 2. Run the preparation pipeline

Run the stages in order when you intend to rebuild the processed data. The default is off so opening or executing the notebook does not overwrite existing outputs.

In [ ]:
runPreparation = False
if runPreparation:
    runDataPreparation()
    runBasePreprocessing()
    attachProfileTargets()
    prepareProfileArrays()
    runSalaryCapNormalization()
print('Preparation pipeline enabled: ' + str(runPreparation))